In [ ]:
# Plantilla de Código: Aprendizaje por Refuerzo con Q-Learning
# Autor: Grok
# Fecha: Noviembre 2025
# Descripción: Esta plantilla implementa Q-Learning en un entorno de Gymnasium.
# El agente aprende a maximizar recompensas explorando y explotando acciones.
# Instrucciones: Reemplaza placeholders (ej. entorno) con tus necesidades.

import gymnasium as gym  # Biblioteca para entornos de RL
import numpy as np
import time  # Para pausas en visualización

# 1. Configuración del Entorno
# Ejemplos de entornos: 'FrozenLake-v1' (grid simple), 'CartPole-v1' (equilibrio), 'Taxi-v3' (taxi).
env_name = 'FrozenLake-v1'  # REEMPLAZA AQUÍ con tu entorno (ej. 'CartPole-v1')
env = gym.make(env_name, render_mode='human')  # 'human' para visualización; usa 'rgb_array' para headless

# Parámetros de Q-Learning
learning_rate = 0.8  # Alfa: cuánto actualizar Q-values (0-1)
discount_factor = 0.95  # Gamma: importancia de recompensas futuras (0-1)
exploration_rate = 1.0  # Epsilon: probabilidad de acción aleatoria (inicia en 1, decae)
min_exploration_rate = 0.01  # Epsilon mínimo
exploration_decay = 0.995  # Decaimiento de epsilon por episodio
num_episodes = 1000  # Número de episodios de entrenamiento

# Inicializar la tabla Q (estados x acciones)
state_space = env.observation_space.n if isinstance(env.observation_space, gym.spaces.Discrete) else env.observation_space.shape[0]
action_space = env.action_space.n
q_table = np.zeros((state_space, action_space))  # Tabla Q inicial en ceros

print(f"Entorno: {env_name} | Estados: {state_space} | Acciones: {action_space}")

# 2. Entrenamiento con Q-Learning
rewards = []  # Para rastrear recompensas por episodio

for episode in range(num_episodes):
    state, _ = env.reset()  # Reiniciar entorno
    total_reward = 0
    done = False
    
    while not done:
        # Exploración vs. Explotación (Epsilon-Greedy)
        if np.random.rand() < exploration_rate:
            action = env.action_space.sample()  # Acción aleatoria (explorar)
        else:
            action = np.argmax(q_table[state])  # Mejor acción conocida (explotar)
        
        # Ejecutar acción
        next_state, reward, done, _, _ = env.step(action)
        
        # Actualizar Q-value (fórmula de Bellman)
        q_table[state, action] = q_table[state, action] + learning_rate * (
            reward + discount_factor * np.max(q_table[next_state]) - q_table[state, action]
        )
        
        state = next_state
        total_reward += reward
        
        # Opcional: Renderizar (visualizar) el entorno
        env.render()
        time.sleep(0.01)  # Pausa para ver (ajusta o quita)
    
    # Decaimiento de exploración
    exploration_rate = max(min_exploration_rate, exploration_rate * exploration_decay)
    
    rewards.append(total_reward)
    if (episode + 1) % 100 == 0:
        print(f"Episodio {episode + 1}/{num_episodes} | Recompensa: {total_reward} | Epsilon: {exploration_rate:.2f}")

env.close()  # Cerrar entorno después de entrenamiento

# 3. Evaluación del Agente Entrenado
# Probar el agente aprendido (sin exploración)
num_test_episodes = 10
test_rewards = []

for _ in range(num_test_episodes):
    state, _ = env.reset()
    total_reward = 0
    done = False
    
    while not done:
        action = np.argmax(q_table[state])  # Siempre la mejor acción
        next_state, reward, done, _, _ = env.step(action)
        state = next_state
        total_reward += reward
        
        env.render()  # Visualizar prueba
        time.sleep(0.5)  # Pausa para ver
    
    test_rewards.append(total_reward)

env.close()

print(f"Recompensa promedio en pruebas: {np.mean(test_rewards):.2f}")

# 4. Visualización de Resultados (Opcional)
import matplotlib.pyplot as plt

plt.figure(figsize=(10, 5))
plt.plot(rewards)
plt.xlabel('Episodios')
plt.ylabel('Recompensa Total')
plt.title('Progreso de Entrenamiento en Q-Learning')
plt.show()

# 5. Extensión Opcional: Guardar/Cargar Q-Table
# np.save('q_table.npy', q_table)  # Guardar
# q_table = np.load('q_table.npy')  # Cargar

# Fin de la plantilla. ¡Experimenta cambiando parámetros o entornos!

In [ ]:
import numpy as np, gymnasium as gym

# ===== Config =====
ENV_ID   = "CartPole-v1"       # "MountainCar-v0", "Acrobot-v1", etc.
SEED     = 42
EPISODES = 800
GAMMA    = 0.99                # descuento (importancia del futuro)
ALPHA    = 0.1                 # tasa de aprendizaje
EPS_INIT = 1.0; EPS_MIN = 0.02; EPS_DECAY = 0.995  # epsilon decay

# ===== Entorno =====
env = gym.make(ENV_ID)
obs_low  = env.observation_space.low
obs_high = env.observation_space.high
n_actions = env.action_space.n

# Algunos entornos tienen límites +/-inf → fijar clips razonables
clip_low  = np.where(np.isfinite(obs_low),  obs_low,  -4.0)
clip_high = np.where(np.isfinite(obs_high), obs_high,  4.0)

# ===== Discretización (elige bins por dimensión) =====
# CartPole: 4 dims; Acrobot: 6; MountainCar: 2
BINS = (6, 6, 12, 12)[:len(clip_low)]  # ajusta si tu entorno tiene otra dimensión
bin_edges = [np.linspace(clip_low[i], clip_high[i], BINS[i]-1) for i in range(len(BINS))]

def discretize(obs):
    obs = np.clip(obs, clip_low, clip_high)
    idxs = [int(np.digitize(obs[i], bin_edges[i])) for i in range(len(BINS))]
    return tuple(idxs)

# ===== Q-table =====
Q = np.zeros(tuple(b+1 for b in BINS) + (n_actions,), dtype=np.float32)

def epsilon_greedy(q_row, epsilon):
    if np.random.rand() < epsilon:
        return np.random.randint(n_actions)           # explorar
    return int(np.argmax(q_row))                      # explotar

# ===== Entrenamiento =====
epsilon = EPS_INIT
rng = np.random.default_rng(SEED)

for ep in range(1, EPISODES+1):
    obs, _ = env.reset(seed=SEED+ep)
    s = discretize(obs)
    done = False
    while not done:
        a = epsilon_greedy(Q[s], epsilon)
        obs2, r, term, trunc, _ = env.step(a)
        done = term or trunc
        s2 = discretize(obs2)

        # --- Q-learning update ---
        td_target = r + (0.0 if done else GAMMA * np.max(Q[s2]))
        Q[s][a]  += ALPHA * (td_target - Q[s][a])

        s = s2

    # decaimiento epsilon por episodio (puede ser por paso si quieres)
    epsilon = max(EPS_MIN, epsilon * EPS_DECAY)

# ===== Evaluación (greedy) =====
def evaluate(n_episodes=10):
    rewards = []
    for _ in range(n_episodes):
        obs, _ = env.reset()
        s = discretize(obs); done = False; G = 0
        while not done:
            a = int(np.argmax(Q[s]))
            obs, r, term, trunc, _ = env.step(a)
            s = discretize(obs); done = term or trunc; G += r
        rewards.append(G)
    return np.mean(rewards)

print("Return medio (greedy):", evaluate())
env.close()


In [ ]:
import numpy as np, gymnasium as gym

# ===== Config =====
ENV_ID   = "CartPole-v1"       # "MountainCar-v0", "Acrobot-v1", etc.
SEED     = 42
EPISODES = 800
GAMMA    = 0.99                # descuento (importancia del futuro)
ALPHA    = 0.1                 # tasa de aprendizaje
EPS_INIT = 1.0; EPS_MIN = 0.02; EPS_DECAY = 0.995  # epsilon decay

# ===== Entorno =====
env = gym.make(ENV_ID)
obs_low  = env.observation_space.low
obs_high = env.observation_space.high
n_actions = env.action_space.n

# Algunos entornos tienen límites +/-inf → fijar clips razonables
clip_low  = np.where(np.isfinite(obs_low),  obs_low,  -4.0)
clip_high = np.where(np.isfinite(obs_high), obs_high,  4.0)

# ===== Discretización (elige bins por dimensión) =====
# CartPole: 4 dims; Acrobot: 6; MountainCar: 2
BINS = (6, 6, 12, 12)[:len(clip_low)]  # ajusta si tu entorno tiene otra dimensión
bin_edges = [np.linspace(clip_low[i], clip_high[i], BINS[i]-1) for i in range(len(BINS))]

def discretize(obs):
    obs = np.clip(obs, clip_low, clip_high)
    idxs = [int(np.digitize(obs[i], bin_edges[i])) for i in range(len(BINS))]
    return tuple(idxs)

# ===== Q-table =====
Q = np.zeros(tuple(b+1 for b in BINS) + (n_actions,), dtype=np.float32)

def epsilon_greedy(q_row, epsilon):
    if np.random.rand() < epsilon:
        return np.random.randint(n_actions)           # explorar
    return int(np.argmax(q_row))                      # explotar

# ===== Entrenamiento =====
epsilon = EPS_INIT
rng = np.random.default_rng(SEED)

for ep in range(1, EPISODES+1):
    obs, _ = env.reset(seed=SEED+ep)
    s = discretize(obs)
    done = False
    while not done:
        a = epsilon_greedy(Q[s], epsilon)
        obs2, r, term, trunc, _ = env.step(a)
        done = term or trunc
        s2 = discretize(obs2)

        # --- Q-learning update ---
        td_target = r + (0.0 if done else GAMMA * np.max(Q[s2]))
        Q[s][a]  += ALPHA * (td_target - Q[s][a])

        s = s2

    # decaimiento epsilon por episodio (puede ser por paso si quieres)
    epsilon = max(EPS_MIN, epsilon * EPS_DECAY)

# ===== Evaluación (greedy) =====
def evaluate(n_episodes=10):
    rewards = []
    for _ in range(n_episodes):
        obs, _ = env.reset()
        s = discretize(obs); done = False; G = 0
        while not done:
            a = int(np.argmax(Q[s]))
            obs, r, term, trunc, _ = env.step(a)
            s = discretize(obs); done = term or trunc; G += r
        rewards.append(G)
    return np.mean(rewards)

print("Return medio (greedy):", evaluate())
env.close()


In [ ]:
# Elegir acción siguiente con la misma política (on-policy)
a2 = epsilon_greedy(Q[s2], epsilon)
td_target = r + (0.0 if done else GAMMA * Q[s2][a2])
Q[s][a]  += ALPHA * (td_target - Q[s][a])
a = a2


In [ ]:
def softmax_action(q_row, tau=0.5):
    z = q_row / max(1e-8, tau)
    z = z - z.max()
    p = np.exp(z); p /= p.sum()
    return int(np.random.choice(len(q_row), p=p))


In [ ]:
# K-armed bandit, epsilon-greedy
K = 10; T = 2000
Q = np.zeros(K); N = np.zeros(K)
EPS = 0.1
for t in range(1, T+1):
    a = np.random.randint(K) if np.random.rand() < EPS else int(np.argmax(Q))
    r = pull(a)                      # función oculta del ejercicio
    N[a] += 1
    Q[a] += (r - Q[a]) / N[a]        # media incremental
 